# dynamicsyntax — short tutorial

This notebook mirrors the [README](README.md) examples: grammars, parsing, semantics, LaTeX export, and optional Manim scene generation.

**Prerequisite:** a Python ≥3.13 environment with `dynamicsyntax` installed (e.g. `uv pip install -e .` from the repo root). Optional extras: LaTeX (`latexmk` / `pdflatex`) for compiling `.tex`, Poppler (`pdftoppm`) or ImageMagick for PNG previews, and `uv pip install -e ".[video]"` plus FFmpeg for Manim.

Later cells use **`IPython.display`** to show **inline PNGs** (from LaTeX) and an **embedded MP4** (from Manim) when those tools succeed.

In [6]:
from pathlib import Path
import importlib.util
import shutil
import tempfile

from IPython.display import Image, Video, display

import dynamicsyntax as ds

# All generated files go here (under the OS temp directory for this kernel session).
EXAMPLES_DIR = Path(tempfile.mkdtemp(prefix="dynamicsyntax_examples_"))
print("Output directory:", EXAMPLES_DIR.resolve())

print("Grammars:", ds.get_grammars())
print("Datasets (placeholder):", ds.get_datasets())

Output directory: C:\Users\arash\AppData\Local\Temp\dynamicsyntax_examples_h5fktpp5
Grammars: ['2015-english-ttr', 'ttr']
Datasets (placeholder): []


## Parse and inspect results

`ds.parse` returns a **`ParseResult`**: use `.semantics` for the final TTR record, `.vis()` for the same address-order tree text as the GUI, and `.address_order` as a string.

In [7]:
# One-shot parse: grammar id or alias ("ttr" → bundled 2015-english-ttr)
p = ds.parse("a man arrives", "ttr")
print("ok:", p.ok)
print("semantics:", p.semantics)
# p.vis()  # uncomment to print the tree to the notebook log
print("address_order (first 400 chars):\n", p.address_order[:400])

Skipping lexicon line for 'is' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'was' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'were' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'am' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'are' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'Tuesday' template 'proper': expected 3 metavar(s), found 2
Skipping lexicon line for 'casserole' template 'proper': expected 3 metavar(s), found 2
Skipping lexicon line for 'stew' template 'proper': expected 3 metavar(s), found 2
Skipping lexicon line for 'what' template 'pron_whq': expected 1 metavar(s), found 2
Skipping lexicon line for 'where' template 'pron_whq': expected 1 metavar(s), found 2
Skipping lexicon line for 'what' template 'pron_whq_det': expected 1 metavar(s), found 2
Skipping lexicon line for 'which' template 'pron_whq_det': expec

ok: True
semantics: [p2==pres(e0) : t|x0 : e|p0==man(x0) : t|e0==arrive : es|head==e0 : es|p3==subj(e0,x0) : t]
address_order (first 400 chars):
 [0]  ?Ty(t)  |  —
  [00]  Person(s3) | Class(obj) | Ty(e)  |  Fo([x0 : e|p0==man(x0) : t|head==x0 : e])
  [01] *  ?+eval | Ty(e>t) | !  |  Fo(R1^(R1 ++ [e0==arrive : es|head==e0 : es|p3==subj(e0,R1.head) : t]))
    [000]  Ty(cn)  |  Fo([x0 : e|head==x0 : e|p0==man(x0) : t])
    [001]  Ty(cn>e) | !  |  Fo(R^(R ++ [head==R.head : e]))
    [01L]  Ty(t)  |  Fo([head : es|p2==pres(head) : t])
      [00


In [8]:
# Load grammar once, then parse without repeating the grammar argument
ds.load_grammar("ttr")
p2 = ds.parse("a man arrives")
print(p2.semantics)

Skipping lexicon line for 'is' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'was' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'were' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'am' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'are' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'Tuesday' template 'proper': expected 3 metavar(s), found 2
Skipping lexicon line for 'casserole' template 'proper': expected 3 metavar(s), found 2
Skipping lexicon line for 'stew' template 'proper': expected 3 metavar(s), found 2
Skipping lexicon line for 'what' template 'pron_whq': expected 1 metavar(s), found 2
Skipping lexicon line for 'where' template 'pron_whq': expected 1 metavar(s), found 2
Skipping lexicon line for 'what' template 'pron_whq_det': expected 1 metavar(s), found 2
Skipping lexicon line for 'which' template 'pron_whq_det': expec

[p2==pres(e0) : t|x0 : e|p0==man(x0) : t|e0==arrive : es|head==e0 : es|p3==subj(e0,x0) : t]


In [9]:
# From a grammar folder on disk (same as GUI “load folder”)
# from pathlib import Path
# ds.load_grammar(Path(r"C:\path\to\your\grammar-dir"))
# p3 = ds.parse("go to the red box")

## LaTeX: semantics, tree, incremental

## LaTeX: semantics, tree, incremental

`ParseResult.to_latex(kind=...)` builds a full LaTeX document using shipped styles (`dsttr.sty`, `rtrees`, …). Use `write_tex=` to save a `.tex` file. With `compile_tex=True` you need `latexmk` or `pdflatex` on `PATH`; for PNG, `pdftoppm` or ImageMagick.

The next cell tries to compile **tree** and **semantics** figures to PNG and **shows them inline** when the toolchain succeeds.

`parse(..., trace=True)` records per-word tree snapshots for `kind="incremental"`.

You can also call **`TTRRecordType.to_latex()`** on the final semantics for a small math fragment (not the full document wrapper).

In [10]:
p = ds.parse("a man arrives", "ttr")
sem_tex = p.to_latex("semantics", write_tex=EXAMPLES_DIR / "semantics.tex")
assert "\\documentclass" in sem_tex.tex
print("Wrote:", EXAMPLES_DIR / "semantics.tex")
print(sem_tex.tex[:500], "\n...")

Skipping lexicon line for 'is' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'was' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'were' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'am' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'are' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'Tuesday' template 'proper': expected 3 metavar(s), found 2
Skipping lexicon line for 'casserole' template 'proper': expected 3 metavar(s), found 2
Skipping lexicon line for 'stew' template 'proper': expected 3 metavar(s), found 2
Skipping lexicon line for 'what' template 'pron_whq': expected 1 metavar(s), found 2
Skipping lexicon line for 'where' template 'pron_whq': expected 1 metavar(s), found 2
Skipping lexicon line for 'what' template 'pron_whq_det': expected 1 metavar(s), found 2
Skipping lexicon line for 'which' template 'pron_whq_det': expec

Wrote: C:\Users\arash\AppData\Local\Temp\dynamicsyntax_examples_h5fktpp5\semantics.tex
\documentclass{article}
\usepackage[a4paper,margin=1cm]{geometry}
\usepackage{graphicx}
\providecommand{\arrow}[1]{$\Rightarrow$}
\input{dsttr.sty}
\begin{document}
\section*{dynamicsyntax — semantics}
\begin{center}
\[\left[\begin{array}{l}p2\mathrel{ : =}pres(e0)  :  t \\ x0  :  e \\ p0\mathrel{ : =}man(x0)  :  t \\ e0\mathrel{ : =}arrive  :  es \\ head\mathrel{ : =}e0  :  es \\ p3\mathrel{ : =}subj(e0,x0)  :  t\end{array}\right]\]
\end{center}
\end{document} 
...


In [11]:
q = ds.parse("a man arrives", "ttr", trace=True)
inc = q.to_latex("incremental", write_tex=EXAMPLES_DIR / "trace.tex")
assert "figure*" in inc.tex
print("Wrote:", EXAMPLES_DIR / "trace.tex")

tree_doc = q.to_latex("tree", write_tex=EXAMPLES_DIR / "tree.tex")
assert "\\begin{tree}" in tree_doc.tex
print("Wrote:", EXAMPLES_DIR / "tree.tex")

Skipping lexicon line for 'is' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'was' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'were' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'am' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'are' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'Tuesday' template 'proper': expected 3 metavar(s), found 2
Skipping lexicon line for 'casserole' template 'proper': expected 3 metavar(s), found 2
Skipping lexicon line for 'stew' template 'proper': expected 3 metavar(s), found 2
Skipping lexicon line for 'what' template 'pron_whq': expected 1 metavar(s), found 2
Skipping lexicon line for 'where' template 'pron_whq': expected 1 metavar(s), found 2
Skipping lexicon line for 'what' template 'pron_whq_det': expected 1 metavar(s), found 2
Skipping lexicon line for 'which' template 'pron_whq_det': expec

Wrote: C:\Users\arash\AppData\Local\Temp\dynamicsyntax_examples_h5fktpp5\trace.tex
Wrote: C:\Users\arash\AppData\Local\Temp\dynamicsyntax_examples_h5fktpp5\tree.tex


In [12]:
# Compile LaTeX to PNG and show images in the notebook (needs latexmk/pdflatex + pdftoppm or magick/convert)


def _show_latex_png(parse_result, *, kind: str, label: str, png_path: Path) -> None:
    """Compile *parse_result*'s LaTeX (*kind*) to *png_path* and display it if tools succeed."""
    if not (shutil.which("latexmk") or shutil.which("pdflatex")):
        print(f"[{label}] Skipped: no latexmk/pdflatex on PATH.")
        return
    pdf_path = png_path.with_suffix(".pdf")
    try:
        out = parse_result.to_latex(
            kind,  # type: ignore[arg-type]
            compile_tex=True,
            image_path=png_path,
            pdf_out=pdf_path,
        )
    except (FileNotFoundError, OSError, RuntimeError) as exc:
        print(f"[{label}] Compile skipped: {exc}")
        return
    if out.png_path and Path(out.png_path).is_file():
        display(Image(filename=str(out.png_path)))
    else:
        print(f"[{label}] No PNG (install pdftoppm or ImageMagick for raster export).")


# `q` / `p` come from the cells above (re-run those cells if needed).
_show_latex_png(q, kind="tree", label="DS tree (rtrees)", png_path=EXAMPLES_DIR / "tree.png")
_show_latex_png(p, kind="semantics", label="TTR semantics", png_path=EXAMPLES_DIR / "semantics.png")

[DS tree (rtrees)] Compile skipped: LaTeX compilation failed (exit 12); install latexmk or pdflatex and ensure dsttr/rtrees dependencies resolve.


KeyboardInterrupt: 

In [ ]:
# LaTeX fragment for the TTR record only (no full document)
if p.ok and p.semantics is not None:
    print(p.semantics.to_latex())

\left[\begin{array}{l}p2\mathrel{ : =}pres(e0)  :  t \\ x0  :  e \\ p0\mathrel{ : =}man(x0)  :  t \\ e0\mathrel{ : =}arrive  :  es \\ head\mathrel{ : =}e0  :  es \\ p3\mathrel{ : =}subj(e0,x0)  :  t\end{array}\right]


## Manim (optional)

Install: `uv pip install -e ".[video]"`. Rendering needs Manim’s usual stack (LaTeX for `MathTex`, FFmpeg, etc.).

The next cell writes scene source with `render=False`. The cell after that tries a **low-quality MP4 render** and **embeds the video** when Manim succeeds.

In [ ]:
# Generate scene code without invoking Manim (requires prior parse with trace=True)
p_tr = ds.parse("a man arrives", "ttr", trace=True)
scene = p_tr.to_manim(render=False, write_scene=EXAMPLES_DIR / "parse_scene.py")
assert "from manim import" in scene.scene_code
print("Wrote:", EXAMPLES_DIR / "parse_scene.py")

Skipping lexicon line for 'is' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'was' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'were' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'am' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'are' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'Tuesday' template 'proper': expected 3 metavar(s), found 2
Skipping lexicon line for 'casserole' template 'proper': expected 3 metavar(s), found 2
Skipping lexicon line for 'stew' template 'proper': expected 3 metavar(s), found 2
Skipping lexicon line for 'what' template 'pron_whq': expected 1 metavar(s), found 2
Skipping lexicon line for 'where' template 'pron_whq': expected 1 metavar(s), found 2
Skipping lexicon line for 'what' template 'pron_whq_det': expected 1 metavar(s), found 2
Skipping lexicon line for 'which' template 'pron_whq_det': expec

Wrote: C:\Users\arash\AppData\Local\Temp\dynamicsyntax_examples_0d37izac\parse_scene.py


In [ ]:
# Render Manim to MP4 and show the video inline (needs manim + LaTeX + ffmpeg)
mp4_path = EXAMPLES_DIR / "parse.mp4"

if shutil.which("manim") is None and importlib.util.find_spec("manim") is None:
    print("Manim not installed; skipping video. Install with: uv pip install -e \".[video]\"")
else:
    try:
        vid = p_tr.to_manim(output_path=mp4_path, quality="l", render=True)
        if vid.video_path and Path(vid.video_path).is_file():
            display(Video(str(vid.video_path), embed=True))
        else:
            print("Manim did not produce a video. exit_code:", vid.exit_code)
            if vid.stderr:
                print("stderr tail:\n", vid.stderr[-1200:])
    except (FileNotFoundError, OSError, RuntimeError) as exc:
        print("Manim render skipped:", exc)

In [1]:
import dynamicsyntax as ds

In [2]:
p = ds.parse("a man arrives", "ttr")

Skipping lexicon line for 'is' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'was' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'were' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'am' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'are' template 'v_aux_be': expected 3 metavar(s), found 2
Skipping lexicon line for 'Tuesday' template 'proper': expected 3 metavar(s), found 2
Skipping lexicon line for 'casserole' template 'proper': expected 3 metavar(s), found 2
Skipping lexicon line for 'stew' template 'proper': expected 3 metavar(s), found 2
Skipping lexicon line for 'what' template 'pron_whq': expected 1 metavar(s), found 2
Skipping lexicon line for 'where' template 'pron_whq': expected 1 metavar(s), found 2
Skipping lexicon line for 'what' template 'pron_whq_det': expected 1 metavar(s), found 2
Skipping lexicon line for 'which' template 'pron_whq_det': expec

In [3]:
print("ok:", p.ok)
print("semantics:", p.semantics)
# p.vis()  # uncomment to print the tree to the notebook log
# print("address_order:\n", p.address_order)

ok: True
semantics: [p2==pres(e0) : t|x0 : e|p0==man(x0) : t|e0==arrive : es|head==e0 : es|p3==subj(e0,x0) : t]


In [4]:
# print(p.get_vocab())

In [5]:
print(p.get_vocab(groupby='alpha'))

Lexicon: 2015-english-ttr
Source: C:\ArashMath\DyLan\dynamicsyntax\src\dynamicsyntax\grammars\2015-english-ttr\lexicon.txt

Load statistics:
  Word entries loaded:    360
  Unique words:           232
  Words failed:           200
  Macros loaded:          9
  Macros failed:          0

Failed words:
  arrested, arrested, arrived, arrive, arrive, arrives, arriving, arrived, arrived, arrive
  arrive, arrives, arriving, arrived, arrived, be, be, is, being, was
  were, been, am, are, begun, begun, begin, begin, begins, beginning
  began, begun, begin, begin, begins, beginning, began, begun, begun, believed
  believe, believe, believes, believing, believed, believed, believe, believe, believes, believing
  believed, believed, believe, believe, believes, believing, believed, believed, can, can
  could, disliked, done, flew, flew, gone, gone, go, go, goes
  going, went, gone, go, go, goes, going, went, gone, give
  give, gives, giving, gave, given, hated, hated, left, left, leave
  leave, le

In [6]:
print(p.get_vocab(groupby="alpha", backend="rich"))

Lexicon source: C:\ArashMath\DyLan\dynamicsyntax\src\dynamicsyntax\grammars\2015-english-ttr\lexicon.txt

Load statistics:
  Word entries loaded:    360
  Unique words:           232
  Words failed:           200 (arrested, arrested, arrived, arrive, arrive, arrives, arriving, arrived, arrived, arrive, arrive, arrives, arriving, arrived, 
arrived, be, be, is, being, was, were, been, am, are, begun, begun, begin, begin, begins, beginning, began, begun, begin, begin, begins, beginning, began, 
begun, begun, believed, believe, believe, believes, believing, believed, believed, believe, believe, believes, believing, believed, believed, believe, believe, 
believes, believing, believed, believed, can, can, could, disliked, done, flew, flew, gone, gone, go, go, goes, going, went, gone, go, go, goes, going, went, 
gone, give, give, gives, giving, gave, given, hated, hated, left, left, leave, leave, leaves, leaving, left, left, leave, leave, leaves, leaving, left, left, 
leave, leave, leaves, leaving, left, left, liked, liked, may, may, read, read, sneezed, snored, started, started, start, start, starts, starting, started, 
started, start, start, starts, starting, started, started, started, stayed, stayed, thought, thought, think, think, thinks, thinking, thought, thought, think, 
think, thinks, thinking, thought, thought, think, think, thinks, thinking, thought, thought, travelled, travel, travel, travels, travelling, travelled, 
travelled, travel, travel, travels, travelling, travelled, travelled, wondered, wonder, wonder, wonders, wondering, wondered, wondered, wonder, wonder, wonders,
wondering, wondered, wondered, wonder, wonder, wonders, wondering, wondered, wondered, man, Tuesday, casserole, stew, what, where, what, what, which, who, how, 
when, where, his, her, the)
  Macros loaded:          9
  Macros failed:          0

.                     ?                     April     London    Monday    
Paris                 a                     a         actually  aeroplane 
aeroplanes            airplane              airplane  airplanes airplanes 
an                    an                    apple     arash     arrest    
arrest                arrest                arrest    arrested  arrested  
arresting             arresting             arrests   arrests   arrive    
arrive                arrived               arrives   arriving  bank      
bank                  banks                 banks     beef      began     
began                 began                 begin     begin     begin     
begin                 begin                 begin     beginning beginning 
beginning             begins                begins    begins    believe   
believe               believed              believes  believing big       
big                   bill                  blue      blue      bluer     
bluest                boat                  boats     book      books     
bought                boy                   boys      brand     brands    
buy                   buy                   buying    buys      by        
by                    can                   can       can       can       
car                   carrot                cook      cooked    cooks     
could                 cousin                did       dislike   dislike   
disliked              dislikes              disliking do        does      
europe                every                 fat       february  finally   
financial-institution financial-institution flew      flew      flies     
flies                 fly                   fly       fly       fly       
flying                flying                from      girl      girl      
girls                 go                    go        go        go        
goes                  goes                  going     going     green     
green                 greener               greenest  group     guy       
hate                  hate                  hate      hate      hated     
hated                 hates                 hates     hating    hating    
he                    her                   herself   him       himself   
hotel                 hotels                i         it        it        
january               jill                  john      knew      know      
know                  knows                 lamb      leased    leave     
leave                 leave                 leave     leaves    leaves    
leaving               leaving               left      left      like      
like                  like                  like      liked     liked     
likes                 likes                 liking    liking    london    
looking               lookingfor            madly     man       mary      
massive               massive               may       me        meet      
meet                  meeting               meets     men       met       
nice                  nice                  no        ok        okay      
on                    paris                 phone     plane     plane     
planes                planes                policeman policemen product   
quickly               read                  read      read      read      
read                  read                  reading   reading   reads     
reads                 really                red       red       redder    
reddest               right                 right     rights    riverside 
riversides            room                  rooms     ruth      samsung   
she                   slowly                sneeze    sneeze    sneezed   
sneezes               sneezing              snore     snore     snored    
snores                snoring               some      start     start     
start                 start                 start     start     started   
started               started               starting  starting  starting  
starts                sta

In [ ]:
print(p.get_vocab(groupby="alpha",))